In [ ]:
# have the original dataframe 
# analyze null values for each using the dataframe
# analyze hit rates for that using the dataframe ` `
# compile a list of valid_indices
# get the final valid dataframe

In [ ]:
import scipy.io 
import numpy as np
import pandas as pd
import glob
import matplotlib.pyplot as plt
import pickle as pl
from collections import OrderedDict
import traceback
import random
import threading
import math
import os

In [ ]:
# we shoudl see how much vt is actually covering

### Exploring the structure one mat file

#### Validating vt, birth time , age death 
##### first valdiate that the vt[-1] and vt[0] is within 0, 86,400*age_of_death 
##### see the idfference 64,000*age_of_death  - vt[-1] 
##### since we care about < 1, <3 , <5 day, its better to see the gap between vt[-1] and age of death rather than birht time and vt[0]

In [ ]:
# Validate timestamps and calculate difference from death
validation_results = []

for file_path in mat_files[:50]:
    mat_data = scipy.io.loadmat(file_path)
    baby_id = file_path.split('/')[-1].split('_')[1]
    
    # Get timestamps
    vt = mat_data['vt'].flatten()
    vt_first = vt[0]
    vt_last = vt[-1]
    
    # Get birth time and age at death
    birth_time = mat_data['pdata'][0][1]
    age_at_death = mat_data['pdata'][0][-1]
    
    # Calculate expected max timestamp (64000 seconds/day * age in days)
    expected_max = 86400 * age_at_death if not np.isnan(age_at_death) else np.nan
    
    # Calculate difference: how much time BEFORE death did measurements stop
    # difference = expected_max - vt_last if not np.isnan(expected_max) else np.nan
    difference = vt_first-birth_time 
    
    # Validation: INVALID if vt_first < birth_time OR difference < 0
    is_valid = True
    if not np.isnan(birth_time) and vt_first < birth_time:
        is_valid = False  # Started before birth
    if not np.isnan(difference) and difference < 0:
        is_valid = False  # Continued after death
    
    validation_results.append({
        'baby_id': baby_id,
        'difference': difference,
        'is_valid': is_valid
    })

# Create DataFrame
validation_df = pd.DataFrame(validation_results)

# Summary
print(f"Total files: {len(validation_df)}")
print(f"Valid files: {validation_df['is_valid'].sum()}")
print(f"Invalid files: {(~validation_df['is_valid']).sum()}")
print(f"\nDifference statistics (time before death):")
print(validation_df['difference'].describe())

# Show invalid files
print(f"\nInvalid files:")
print(validation_df[~validation_df['is_valid']][['baby_id', 'difference', 'is_valid']])

In [ ]:
validation_df['difference'].value_counts()

In [ ]:
validation_df

In [ ]:
# The sample data doesnt have any useful things to use
# pname and pdata are metadata of the patient id 
# v data and v name are the vitals and the values
mat_file = scipy.io.loadmat('/Users/adityagoyal/Desktop/Research - yin li/timeseries_project/timeseries_with_chronos/data/download/Vitals/NICU_1018_vitals.mat')

In [ ]:
mat_file.keys()

In [ ]:
mat_file['pname']

In [ ]:
mat_file['pdata']

In [ ]:
columns= [str(i[0]) for i in list(mat_file['pname'].reshape(-1))]
print(columns)

In [ ]:
mat_file['vt']

In [ ]:
mat_file['pname']

In [ ]:
print(mat_file['pdata'])

In [ ]:
mat_file['vname']

In [ ]:
mat_file['vdata']

In [ ]:
valid_summary_dic[valid_summary_dic["indicator_vector"]==1]

### Creating the Patient data from the samples data and the vitals metadata 

In [ ]:
mat_files = sorted(glob.glob('./download/Vitals/*.mat'))

In [ ]:
baby_ids = [file.split('/')[-1].split('_')[1] for file in mat_files]

In [ ]:
dataframes = [ pd.DataFrame(scipy.io.loadmat(file)['pdata'],columns=columns) for file in mat_files]

In [ ]:
metadata_df = pd.concat(dataframes,ignore_index=True)
metadata_df = metadata_df.rename(columns={'PatientID': 'baby_id'})

In [ ]:
with open('metadata_df.pkl','wb') as f:
    pl.dump(f)

In [ ]:
# Question:
# see the null and non-null distribution of 
# id column and the data columns
# (to validate if hr and spo2 are only important) across all mat files

# Analysis:
# Seeing how many null values are there

In [ ]:
# # to see distribution and null percentage of one mat file
# for i in range(data.shape[1]):
#     plt.subplot(1, 3, 1)
#     plt.hist(data[:,i].flatten(), bins=50, alpha=0.7)
#     plt.title(f'{names[i]}')
#     plt.ylabel('Frequency')
#     plt.tight_layout()
#     plt.show()
    
# print("Null values in data columns:")
# for i in range(data.shape[1]):
#     null_count = np.isnan(data[:, i]).sum()
#     total_count = len(data[:, i])
#     print(f"{names[i]}: {null_count} nulls out of {total_count} ({null_count/total_count*100:.1f}%)")

In [ ]:
# Question: 
# see the null and non-null distribution of time_columns values 

# Analysis:
# some values are negative 
# the seconds don't start from (aka measurement doesnt start from midnigt when the baby is born)
# the frequency of measurements within intervals were erratic 

### Saving the context size, birth time and age_at_death arrays

In [ ]:
context_size_arr = []
for file in mat_files:
    time_values = scipy.io.loadmat(file)['vt'].reshape(-1) 
    context_size_arr.append(time_values[-1]-time_values[0])

In [ ]:
age_at_death_arr = [scipy.io.loadmat(file)['pdata'][0][-1] for file in mat_files ]

In [ ]:
birth_time_arr = [scipy.io.loadmat(file)['pdata'][0][1] for file in mat_files ]

In [ ]:
with open('context_size.pkl','wb') as f:
    pl.dump(context_size_arr,f)

In [ ]:
with open('age_at_death.pkl','wb') as f:
    pl.dump(age_at_death_arr,f)

In [ ]:
with open('birth_time.pkl','wb') as f:
    pl.dump(birth_time_arr,f)

#### see the no of files which perish  and see the no of files which do not perish 

In [ ]:
with open('./age_at_death.pkl','rb') as f:
    arr = pl.load(f)

In [ ]:
print("Total Count: " + str(len(arr)))
nan_count = np.isnan(arr).sum()
print(f"Non-NICU count: {nan_count}")
print(f"NICU count: {len(arr)-nan_count}")

In [ ]:
indicator_vector = []
for i in arr:
    if(np.isnan(i)):
        indicator_vector.append(0)
    else:
        indicator_vector.append(1)
    

In [ ]:
with open('indicator_vector.pkl', 'wb') as f:
    pl.dump(indicator_vector, f)

#### Seeing the context size distriubution of the context size for NICU and non NICU

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/EDA/context_size.pkl','rb') as f:
    arr_context_size = pl.load(f)

In [ ]:
arr_context_size_nicu = []
arr_context_size_nonnicu = []

for i in range(len(indicator_vector)) :
    if(indicator_vector[i]==0):
        arr_context_size_nonnicu.append(arr_context_size[i])
    else:
        arr_context_size_nicu.append(arr_context_size[i])

In [ ]:
# seeing the distribution of the values 
plt.hist(arr_context_size_nicu , bins=10, edgecolor='black')
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.title('Histogram of Values')
plt.xlim(0, 3.5e+7) 
plt.grid(True)
plt.show()

### Analysing birth time pkl file

In [ ]:
with open('birth_time.pkl','rb') as f:
    birth_time_values = pl.load(f)

In [ ]:
print("Birth values:")
print(f"  Null:     {np.isnan(birth_time_values).sum():>5} ({np.isnan(birth_time_values).mean()*100:>6.2f}%)")
print(f"  Non-null: {(~np.isnan(birth_time_values)).sum():>5} ({(~np.isnan(birth_time_values)).mean():>6.2f}%)\n")

In [ ]:
# within every 5 minute interval after 0 and until ttd, how many datapoints are complete.(100% window coverage)(atleast one valid datapoint in every window)
# within every 5 minutes interval after first observation , and until the last observation, how many datapoints are complete(100% window coverage)(atleast one valid datapoint in every window)

# the ttd is not given and from the readme, it seems the last observation doesnt coincide the ttd..but first we need to find the first ttd
# np.float64(0.9524762381190596) of the datapoitns have there ageatdeath as NaN

In [ ]:
# What wre we lloking for ? one of the two options?is it till startobserv,last observ or 0,ttd
# what is a valid datapoint for us ? do we include resp as well along with the cols we think?

In [ ]:
# x1,y1
# x2,y2
# sum(totalx,totaly)

### checking the gap between the last timestamp and age at death 

In [ ]:
# open every mat file, see if its label = 1
# check that the timestamp column exists
# take the last   (age_at_death*86400 + birth time) - ( last_timestamp + birth time ) = age_at_death*86400 - last_timestamp 
# store it

In [ ]:
mat_data = scipy.io.loadmat('/Users/adityagoyal/Desktop/Research - yin li/timeseries_project/timeseries_with_chronos/data/download/Vitals/NICU_1011_vitals.mat')
mat_data

In [ ]:
pdata = mat_data["pdata"][0]
pname = [p[0] if hasattr(p, "__len__") else p for p in mat_data["pname"].flat]

In [ ]:
## storing the relevant information 

import glob
import pickle
import numpy as np
import scipy.io
from concurrent.futures import ThreadPoolExecutor, as_completed

MAT_PATTERN = "../data/download/Vitals/*.mat"
CHECKPOINT_EVERY = 10
PKL_PATH = "death_gap_results.parquet"

# birthtime 
# age_at_death (signifies whether the person died or not)
# first timestamp
# last timestamp

# have birthtime 
# have non null vt

# we are including files with no valid hr and spo2 as well, we are nto filtering for that criteria , as we are jsut lookign for 
# the gap between the last timestamp and death

def one_file(path):
    
    try:
        m = scipy.io.loadmat(path)
        pdata = m["pdata"][0]
        pname = [p[0] if hasattr(p, "__len__") else p for p in m["pname"].flat]

        birth = float(pdata[pname.index("BirthTime")])
        age_at_death = float(pdata[pname.index("Age at Death")])

        if np.isnan(age_at_death):
            age_at_death = None

        if "vt" not in m:
            raise ValueError("no vt")

        first_ts = float(np.asarray(m["vt"]).flat[0])
        last_ts = float(np.asarray(m["vt"]).flat[-1])
        baby_id = path.split('/')[-1].split('_')[1]
        
        return {
            "baby_id": baby_id, 
            "birthtime": birth,
            "age_at_death": age_at_death,
            "first_ts": first_ts,
            "last_ts": last_ts
            }
    except Exception as E:
        raise E


def save_pkl(rows):
    df = pd.DataFrame(rows)
    df.to_parquet(PKL_PATH)
    print(f"saved {len(rows)} rows -> {PKL_PATH}")


def main():
    paths = sorted(glob.glob(MAT_PATTERN))
    rows = []

    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = {pool.submit(one_file, p): p for p in paths}
        for fut in as_completed(futures):
            try:
                row = fut.result()
                if row is not None:
                    rows.append(row)
                    if len(rows) % CHECKPOINT_EVERY == 0:
                        save_pkl(rows)
                    if(len(rows)==10):
                        exit(0)
            except Exception as e:
                print(e)
                print(f"skip {futures[fut]}: {e}")

    save_pkl(rows)  # final save
    print(f"done: {len(rows)} deceased babies, {done} files scanned")
    return rows


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_project/timeseries_with_chronos/EDA/data/death_gap_results.parquet','rb') as f:
    df = pd.read_parquet(f)

In [ ]:
df = df.loc[df["age_at_death"].notna()]
df["context_size_recorded"] = df["last_ts"] - df["first_ts"]
df["context_size_actual"] = df["age_at_death"] * 86400
df["death_gap_seconds"] = ( df["age_at_death"] * 86400) - df["last_ts"]

In [ ]:
df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# CONFIG
# =============================================================================

PARQUET_PATH = "data/death_gap_results.parquet"  # change to your path

N_BIRTHTIME_BINS = 8

RECORDED_CONTEXT_BIN_EDGES = [
    0,
    1 * 86400,
    3 * 86400,
    80 * 3600,   # ~3.3 days — must be before 7d
    7 * 86400,
    14 * 86400,
    np.inf,
]

ACTUAL_CONTEXT_BIN_EDGES = [
    0,
    1 * 86400,
    3 * 86400,
    7 * 86400,
    14 * 86400,
    30 * 86400,
    np.inf,
]


def bin_labels(edges):
    labels = []
    for i in range(len(edges) - 1):
        a, b = edges[i], edges[i + 1]
        if np.isfinite(b):
            labels.append(f"{a/86400:.1f}–{b/86400:.1f}d")
        else:
            labels.append(f"≥{a/86400:.1f}d")
    return labels


# =============================================================================
# LOAD + COLUMNS  (first_ts / last_ts = seconds since birth)
# =============================================================================

df = pd.read_parquet(PARQUET_PATH)

df["context_size_recorded"] = df["last_ts"] - df["first_ts"]
df["context_size_actual"] = df["age_at_death"] * 86400
df["death_gap_seconds"] = df["age_at_death"] * 86400 - df["last_ts"]

eda = df.loc[df["age_at_death"].notna() & df["death_gap_seconds"].notna()].copy()


# =============================================================================
# HELPERS
# =============================================================================

def summarize_gap(gaps, title=""):
    s = pd.Series(gaps).dropna()
    print(f"\n=== {title} (n={len(s)}) ===")
    if len(s) == 0:
        return
    print(f"mean:   {s.mean():12.1f} s  ({s.mean()/86400:.3f} d)")
    print(f"median: {s.median():12.1f} s  ({s.median()/86400:.3f} d)")
    print(f"std:    {s.std():12.1f} s  ({s.std()/86400:.3f} d)")
    print(f"min: {s.min():.1f}  max: {s.max():.1f}")
    for p in (5, 25, 50, 75, 95):
        v = np.percentile(s, p)
        print(f"p{p}: {v:.1f} s ({v/86400:.3f} d)")


def gap_by_group_table(data, group_col):
    t = data.groupby(group_col, observed=True)["death_gap_seconds"].agg(
        n="count", mean="mean", median="median", std="std"
    )
    t["mean_d"] = t["mean"] / 86400
    t["median_d"] = t["median"] / 86400
    return t


def plot_gap_histogram(gaps, title, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))
    ax.hist(pd.Series(gaps).dropna() / 86400, bins=40, edgecolor="k", alpha=0.75)
    ax.set_xlabel("death_gap_seconds (days)")
    ax.set_ylabel("count")
    ax.set_title(title)
    return ax


def heatmap_median_gap(data, row_col, col_col, title=""):
    pivot = data.pivot_table(
        values="death_gap_seconds",
        index=row_col,
        columns=col_col,
        aggfunc="median",
        observed=True,
    )
    fig, ax = plt.subplots(figsize=(max(7, pivot.shape[1]), max(5, pivot.shape[0] * 0.6)))
    im = ax.imshow(pivot.values, aspect="auto", origin="lower")
    ax.set_xticks(np.arange(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(pivot.shape[0]))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel(col_col)
    ax.set_ylabel(row_col)
    ax.set_title(f"{title} (median gap)")
    plt.colorbar(im, ax=ax, label="seconds")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v/86400:.1f}d", ha="center", va="center", fontsize=7)
    plt.tight_layout()
    return pivot, fig


# =============================================================================
# BINS
# =============================================================================

birth_edges = np.unique(np.quantile(eda["birthtime"], np.linspace(0, 1, N_BIRTHTIME_BINS + 1)))
eda["birthtime_bin"] = pd.cut(eda["birthtime"], bins=birth_edges, include_lowest=True)

eda["recorded_context_bin"] = pd.cut(
    eda["context_size_recorded"],
    bins=RECORDED_CONTEXT_BIN_EDGES,
    labels=bin_labels(RECORDED_CONTEXT_BIN_EDGES),
    include_lowest=True,
)

eda["actual_context_bin"] = pd.cut(
    eda["context_size_actual"],
    bins=ACTUAL_CONTEXT_BIN_EDGES,
    labels=bin_labels(ACTUAL_CONTEXT_BIN_EDGES),
    include_lowest=True,
)


# =============================================================================
# 1) OVERALL + BY GLOBAL BIRTHTIME
# =============================================================================

summarize_gap(eda["death_gap_seconds"], "Overall death gap")

plt.figure(figsize=(7, 4))
plot_gap_histogram(eda["death_gap_seconds"], "Overall gap distribution")
plt.tight_layout()
plt.show()

print("\nGap by birthtime bin:")
display(gap_by_group_table(eda, "birthtime_bin"))

groups = list(eda.groupby("birthtime_bin", observed=True))
ncols, nrows = 4, int(np.ceil(len(groups) / 4))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3 * nrows))
for ax, (name, sub) in zip(axes.flat, groups):
    plot_gap_histogram(sub["death_gap_seconds"], str(name), ax=ax)
for ax in axes.flat[len(groups):]:
    ax.axis("off")
fig.suptitle("Gap distribution per birthtime bin")
plt.tight_layout()
plt.show()


# =============================================================================
# 2) BY RECORDED CONTEXT (last_ts - first_ts)
# =============================================================================

print("\nGap by recorded context bin:")
display(gap_by_group_table(eda, "recorded_context_bin"))

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (name, sub) in zip(axes.flat, eda.groupby("recorded_context_bin", observed=True)):
    plot_gap_histogram(sub["death_gap_seconds"], str(name), ax=ax)
fig.suptitle("Gap vs recorded context size")
plt.tight_layout()
plt.show()


# =============================================================================
# 3) BY ACTUAL CONTEXT (age_at_death * 86400)
# =============================================================================

print("\nGap by actual (lifespan) context bin:")
display(gap_by_group_table(eda, "actual_context_bin"))

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (name, sub) in zip(axes.flat, eda.groupby("actual_context_bin", observed=True)):
    plot_gap_histogram(sub["death_gap_seconds"], str(name), ax=ax)
fig.suptitle("Gap vs actual context (lifespan)")
plt.tight_layout()
plt.show()


# =============================================================================
# 4) 2D HEATMAPS (median gap)
# =============================================================================

heatmap_median_gap(eda, "actual_context_bin", "birthtime_bin", "Actual context × birthtime")
heatmap_median_gap(eda, "recorded_context_bin", "birthtime_bin", "Recorded context × birthtime")
plt.show()

print("Cell counts — actual context × birthtime:")
display(
    eda.pivot_table(
        index="actual_context_bin",
        columns="birthtime_bin",
        values="death_gap_seconds",
        aggfunc="count",
        observed=True,
    )
)

In [ ]:
"""Analyze death_gap_results.pkl — stats + histogram + vs context sizes."""

import pickle
import numpy as np
import matplotlib.pyplot as plt

PARQUET_PATH = "death_gap_results.pkl"
HIST_PATH = "death_gap_hist.png"

# Name -> length in seconds (edit this list only)
CONTEXT_SIZE_SECONDS = {
    3 * 86400,
    5 * 86400,
    7 * 86400,
    80 * 86400,
}


def load_gaps(parquet_path=PARQUET_PATH):
    with open(parquet_path, "rb") as f:
        data = pd.read_parquet(f)
    return np.asarray(data["gap_seconds"], dtype=float)


def gaps_for_context_size(gaps, C):
    """Babies whose last sample is at most C seconds before death."""
    gaps = np.asarray(gaps, dtype=float)
    return gaps[gaps <= C]



def print_distribution_stats(gaps):
    """Mean, median, std and a few example percentiles."""
    gaps = np.asarray(gaps, dtype=float)
    print(f"n = {len(gaps)}")
    print(f"mean:   {gaps.mean():.2f} s  ({gaps.mean() / 86400:.3f} days)")
    print(f"median: {np.median(gaps):.2f} s  ({np.median(gaps) / 86400:.3f} days)")
    print(f"std:    {gaps.std():.2f} s  ({gaps.std() / 86400:.3f} days)")
    print(f"min: {gaps.min():.2f}  max: {gaps.max():.2f}")
    for p in (5, 25, 50, 75, 95):
        v = np.percentile(gaps, p)
        print(f"p{p}: {v:.2f} s ({v / 86400:.3f} days)")


def print_vs_context_sizes(gaps, context_sizes_seconds):
    """
    For each context length C (seconds):
    gap > C  → last vital more than C seconds before death.
    """
    gaps = np.asarray(gaps, dtype=float)
    n = len(gaps)
    print("\nContext size checks (gap > C):")
    count = int(np.sum(gaps > context_sizes_seconds))
    gaps = gaps_for_context_size(gaps, context_sizes_seconds)
    print_distribution_stats(gaps)

def plot_gap_distribution(gaps, out_path=HIST_PATH):
    gaps_days = np.asarray(gaps, dtype=float) / 86400
    plt.figure(figsize=(8, 4))
    plt.hist(gaps_days, bins=40, edgecolor="k", alpha=0.75)
    plt.xlabel("Death gap (days): age_at_death×86400 − last vt")
    plt.ylabel("Count")
    plt.title("Death time gap (label=1 babies)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"wrote {out_path}")

def plot_gap_distribution_per_context_size(gaps, context_sizes_seconds, out_path="death_gap_hist_per_context.png"):
    gaps = np.asarray(gaps, dtype=float)
    names = list(context_sizes_seconds.keys())
    n = len(names)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), squeeze=False)

    for ax, name in zip(axes.flat, names):
        C = context_sizes_seconds[name]
        sub = gaps_for_context_size(gaps, C)
        if len(sub) == 0:
            ax.set_title(f"{name}: empty")
            continue
        ax.hist(sub / 86400, bins=30, edgecolor="k", alpha=0.75)
        ax.set_xlabel("Gap (days)")
        ax.set_ylabel("Count")
        ax.set_title(f"{name}\ngap ≤ {C/86400:.1f}d  (n={len(sub)})")

    fig.suptitle("Death gap distribution per context size", y=1.02)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"wrote {out_path}")
    
    
def run_full_analysis(parquet_path=PARQUET_PATH):
    gaps = load_gaps(parquet_path)
    print("=== Death gap stats ===") 
    print_distribution_stats(gaps)
    print_vs_context_sizes(gaps, CONTEXT_SIZE_SECONDS)
    plot_gap_distribution(gaps)
    return gaps


if __name__ == "__main__":
    run_full_analysis()

In [ ]:
def gaps_for_context_size(gaps, C):
    """Babies whose last sample is at most C seconds before death."""
    gaps = np.asarray(gaps, dtype=float)
    return gaps[gaps <= C]


def print_stats_per_context_size(gaps, context_sizes_seconds):
    gaps = np.asarray(gaps, dtype=float)
    print("\n=== Gap stats per context size (subset: gap <= C) ===")
    for name, C in context_sizes_seconds.items():
        sub = gaps_for_context_size(gaps, C)
        if len(sub) == 0:
            print(f"{name}: no babies (C={C} s)")
            continue
        print(f"\n{name}  (C={C} s, n={len(sub)}/{len(gaps)})")
        print(f"  mean={sub.mean():.2f} s  median={np.median(sub):.2f} s  std={sub.std():.2f} s")
        print(f"  min={sub.min():.2f}  max={sub.max():.2f}")




#### Checking window coverage for HR and SP02 (vectorized and non-vectorized approach)

In [ ]:
# timestamps: [2,3,5],  hr[null, 3, null]  spo2[null,2,3]
# timestamps - take min and max 
# distribute it step size = 2 
# copy voer the valid values for the timestamps 
# do the analaysis for per 5 time windows (misses,hits,noofwindows)

# store the true, false arrays in pickle files
# hits,no of windows  for this particular window size
# one row for each baby for a particular window size
# cols would be window sizes.

# do this for every baby 

In [ ]:
# take the hr values and see which are non null 
# take the sp2 values and see which are non null 
# do the and to find the values which have non-null hr and sp02
# store this and the timestamps array

# for wsize in sizes:
#   for baby in babies:
        # timestamp is sorted array
        # the first window starts from the first timestamp
        # see if the next value is in the current window, if yes, assign the window a hit
        # see if the next value is in the next window, increment the counter to see the next window by the window size
        # else, see how many middle windows we have missed 
        # append the (no of misses, hits, and total no of windows)

In [ ]:
# take the start and end timestamp to figure out the no of windows

# you only need the timestamp info to be stored later to compute for arbitary window sizes
# since you can just take the timestamp with corresponding true in final_arr
# basiclaly in order to sae compute jsut assign those invalid timestamps to be null

# after getting those valid timestamps, you then iterate through the timestamps and then group timestamps in one window to one timestamp
# start with a timestamp, icnrement the counter
# if timestamp!=null
# see if the next is within the 300 s and skip it 
# once you find one which is not, windowsize - (timestamp - 300*(timestamp-orig)/300)), increment the counter 
# see if the next one is within that range and skip it 

# we have the no of hits, then we have the no of windows, no of misses = no of windows - no of hits

In [ ]:
# we are starting the measurement from a valid timestamp since we need to know where to start the window strides. We cant jsut start from the first global timstamp
# we are assuming the timestamps are non-null and sorted

In [ ]:
# We run both of the window coverages cause we need to verify that each one works and we get the correct hit rate 
# The final df released by these window coverage methods doesn't include files which dont even have HR and SPO2 columns

In [ ]:
mat_files = set(sorted(glob.glob('../data/download/Vitals/*.mat')))

In [ ]:
len(mat_files)

In [ ]:
sorted(mat_files)

In [ ]:
arr = {
    
"../data/download/Vitals/NICU_1169_vitals.mat",
"../data/download/Vitals/NICU_1527_vitals.mat",
"../data/download/Vitals/NICU_3866_vitals.mat",
"../data/download/Vitals/NICU_4933_vitals.mat",
"../data/download/Vitals/NICU_5294_vitals.mat",
"../data/download/Vitals/NICU_6283_vitals.mat",
"../data/download/Vitals/NICU_6647_vitals.mat",
"../data/download/Vitals/NICU_6745_vitals.mat",
"../data/download/Vitals/NICU_7799_vitals.mat",
"../data/download/Vitals/NICU_8761_vitals.mat"
    
}



In [ ]:
result = sorted(list(mat_files - arr))

In [ ]:
len(result)

In [ ]:
# code to check for whether the vname can be hardcoded or not
mat_files = sorted(glob.glob('../data/download/Vitals/*.mat'))
false_num = 0
for i, file_path in enumerate(mat_files):
        mat_data = scipy.io.loadmat(file_path)
        
        try:
             bool1 = mat_data['vname'][0][0][0] == 'HR'
             bool2 = mat_data['vname'][2][0][0] == 'SPO2-%'
        
        except:
             print(file_path)
             print(mat_data['vname'])
                
        final_bool = bool1 and bool2
        if not(final_bool):
                false_num+=1
                

In [ ]:
with open('../data/valid_summary_dic.pkl','rb') as f:
    df = pl.load(f)
    

In [ ]:
len(mat_files)

In [ ]:
mat_files = result

In [ ]:
# ./download/Vitals/NICU_1169_vitals.mat
# ./download/Vitals/NICU_1527_vitals.mat
# ./download/Vitals/NICU_3866_vitals.mat
# ./download/Vitals/NICU_4933_vitals.mat
# ./download/Vitals/NICU_5294_vitals.mat
# ./download/Vitals/NICU_6283_vitals.mat
# ./download/Vitals/NICU_6647_vitals.mat
# ./download/Vitals/NICU_6745_vitals.mat
# ./download/Vitals/NICU_7799_vitals.mat
# ./download/Vitals/NICU_8761_vitals.mat
# dont have either of the columns

In [ ]:
# # Find HR and SPO2 columns
# hr_cols = mat_data['vdata'][:,0]
# spo2_cols = mat_data['vdata'][:,2]
# valid_timestamps = mat_data['vt']

# # Step 1: Filter for non-null HR and SPO2 values
# final_arr = ~np.isnan(hr_cols) & ~np.isnan(spo2_cols)

### drawing the missing rate graph

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_project/timeseries_with_chronos/EDA/results_vectorized.pkl','rb') as f:
    final_df = pd.read_pickle(f)

In [ ]:
final_df

In [ ]:
cols = [col for col in final_df.columns if "coverage" in col]
vals = [final_df[col].mean() for col in cols]
hits = [col for col in final_df.columns if "hits" in col]
total_windows = [col for col in final_df.columns if "total_windows" in col]

In [ ]:
weighted_vals = []
for i in range(len(hits)):
    hit = hits[i]
    window = total_windows[i]
    weighted_vals.append(final_df[hit].sum()/final_df[window].sum())
    print(final_df[hit].sum())
    print(final_df[window].sum())
    

In [ ]:
weighted_vals

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# 1. Raw data
# pls ensure the labels are the same as the data in the hitrate scripts
labels = [
    'coverage_1s', 'coverage_2s', 'coverage_3s', 'coverage_4s', 'coverage_5s', 
    'coverage_10s', 'coverage_20s', 'coverage_24s', 'coverage_32s', 'coverage_44s', 
    'coverage_58s', 'coverage_78s', 'coverage_105s', 'coverage_141s', 'coverage_188s', 
    'coverage_252s', 'coverage_300s', "coverage_1083s", "coverage_1940s", 'coverage_3000s'
]


# hitting_rates = [
#     0.42062409534580203, 0.8412446614411365, 0.8421938899835952, 0.8431360835029188, 
#     0.8438421817210304, 0.8467888369728861, 0.8500083001880581, 0.8509355812759477, 
#     0.8525278249517725, 0.8544598004924647, 0.8561848863408378, 0.8579765891244808, 
#     0.859665799442101, 0.8612399932846905, 0.8625090295770801, 0.8637031730757724, 
#     0.8643615857663562, 0.8752060994528954
# ]

hitting_rates = [np.float64(0.36936859623299906),
 np.float64(0.7387353576152783),
 np.float64(0.739530286061961),
 np.float64(0.7403115964903285),
 np.float64(0.7409072786441004),
 np.float64(0.7433775112244542),
 np.float64(0.7460636767506823),
 np.float64(0.7468498889919469),
 np.float64(0.7481977945352047),
 np.float64(0.7498156290587213),
 np.float64(0.7512665711381438),
 np.float64(0.7527412882128685),
 np.float64(0.7541699356276246),
 np.float64(0.7554235259760453),
 np.float64(0.7564924454410845),
 np.float64(0.7575165114978493),
 np.float64(0.7581009855141552),
 np.float64(0.7630432913682789),
 np.float64(0.7662590860429079),
 np.float64(0.7701378262686414)]


# 2. Extract raw integer second values
seconds = [int(lbl.replace('coverage_', '').replace('s', '')) for lbl in labels]
missing_rates = [1.0 - hr for hr in hitting_rates]

# 3. Create Plot
plt.figure(figsize=(8, 5))
plt.plot(seconds, missing_rates, marker='o', color='b', linestyle='-', linewidth=2)

# Set both axes to log scale
plt.xscale('log')
# plt.yscale('log')
plt.ylim(0,1)

# 4. Force the log-scale axis to display raw numbers instead of 10^x
plt.gca().xaxis.set_major_formatter(ScalarFormatter())
plt.gca().yaxis.set_major_formatter(ScalarFormatter())

# Set custom, clean numeric tick positions so they don't overlap on the log scale
x_ticks = [1, 2, 5, 10, 20, 50, 100, 300, 3000]
plt.xticks(x_ticks, labels=x_ticks)

# NOTE: plt.xlabel(...) is omitted entirely so there is no text label on the X-axis.
plt.ylabel('Missing Rate (1 - Coverage)')
plt.xlabel('sampling rate/window size (seconds)')
plt.title('Missing Rate vs. Time')
plt.grid(True, which="both", ls="--", alpha=0.5)

plt.tight_layout()
plt.savefig('missing_plot_complete_weighted.png', dpi=300, bbox_inches='tight')
plt.show()

## validating vectorized vs non_optimal approach

In [ ]:
with open('../EDA/old/results_non_optimal_fixed.pkl','rb') as f:
    df_non_optimal = pl.load(f)['summary_df']

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_project/timeseries_with_chronos/EDA/results_vectorized_1.pkl','rb') as f:
    df_vectorized = pl.load(f)['summary_df']

In [ ]:
df_vectorized

In [ ]:
columns_to_compare = set(df_non_optimal.columns) & set(df_vectorized.columns) - set("baby_id")

In [ ]:
different_ids = set()
baby_ids_list = list(df_non_optimal['baby_id'])
for column in columns_to_compare:
    for id in baby_ids_list:
        if(df_vectorized[df_vectorized['baby_id']==id][column].values[0] != df_non_optimal[df_non_optimal['baby_id']==id][column].values[0]):
            different_ids.add(id)

In [ ]:
df_non_optimal[df_non_optimal['baby_id'].isin(list(different_ids))]

In [ ]:
df_vectorized[df_vectorized['baby_id'].isin(list(different_ids))]

In [ ]:
# learnings 
# always validate your code , with different approaches 
# follow your gut feeling and alway establish that the assumptions are true...for eg sorted, column positions 
# never do inplace (although helps during memeory bottlenecks)

### Visualizing the hit rate of the summary df

In [ ]:
with open('../EDA/old/results_vectorized_fixed.pkl','rb') as f:
    dic = pl.load(f)

In [ ]:
raw_summary_dic = dic['summary_df']

In [ ]:
coverage_cols = [col for col in raw_summary_dic.columns if 'coverage' in col]
for col in coverage_cols:
    print(f"\n{col}:")
    print(f"  Mean: {raw_summary_dic [col].mean():.4f}")
    print(f"  Median: {raw_summary_dic [col].median():.4f}")
    print(f"  25th percentile: {raw_summary_dic [col].quantile(0.25):.4f}")
    print(f"  75th percentile: {raw_summary_dic [col].quantile(0.75):.4f}")
    print(f"  Min: {raw_summary_dic [col].min():.4f}")
    print(f"  Max: {raw_summary_dic [col].max():.4f}")

In [ ]:
for col in coverage_cols:
    plt.figure(figsize=(4, 4))
    plt.boxplot(raw_summary_dic[col].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2),
                whiskerprops=dict(color='blue', linewidth=1.5),
                capprops=dict(color='blue', linewidth=1.5))
    plt.ylabel('Coverage Rate', fontsize=12)
    plt.title(f'Boxplot of {col}', fontsize=14, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
valid_summary_dic = raw_summary_dic[raw_summary_dic['coverage_60s'] >= 0.70]

In [ ]:
coverage_cols = [col for col in valid_summary_dic.columns if 'coverage' in col]
for col in coverage_cols:
    print(f"\n{col}:")
    print(f"  Mean: {valid_summary_dic[col].mean():.4f}")
    print(f"  Median: {valid_summary_dic[col].median():.4f}")
    print(f"  25th percentile: {valid_summary_dic[col].quantile(0.25):.4f}")
    print(f"  75th percentile: {valid_summary_dic[col].quantile(0.75):.4f}")
    print(f"  Min: {valid_summary_dic[col].min():.4f}")
    print(f"  Max: {valid_summary_dic[col].max():.4f}")

In [ ]:
for col in coverage_cols:
    plt.figure(figsize=(4, 4))
    plt.boxplot(valid_summary_dic[col].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2),
                whiskerprops=dict(color='blue', linewidth=1.5),
                capprops=dict(color='blue', linewidth=1.5))
    plt.ylabel('Coverage Rate', fontsize=12)
    plt.title(f'Boxplot of {col}', fontsize=14, fontweight='bold')
    plt.ylim(0, 1)  
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
baby_id_to_indicator = {}
baby_id_to_context_size = {}

for file_path in mat_files:
    baby_id = file_path.split('/')[-1]
    
    # Load mat file
    mat_data = scipy.io.loadmat(file_path)
    
    # Get age at death
    age_at_death = mat_data['pdata'][0][-1]
    baby_id_to_indicator[baby_id] = 0 if np.isnan(age_at_death) else 1
    
    # Compute context size (time range)
    timestamps = mat_data['vt'].flatten()
    context_size = timestamps[-1] - timestamps[0]
    baby_id_to_context_size[baby_id] = context_size.item()

# Map to dataframe
valid_summary_dic['indicator_vector'] = valid_summary_dic['baby_id'].map(baby_id_to_indicator)
valid_summary_dic['context_size'] = valid_summary_dic['baby_id'].map(baby_id_to_context_size)

In [ ]:
valid_summary_dic[valid_summary_dic['indicator_vector']==0]

### validating the implementation before implementing 

In [ ]:
import pickle as pl
import matplotlib.pyplot as plt
import os
import numpy as np

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_project/timeseries_with_chronos/experiment_files/results_3d_focal.pkl','rb') as f: 
    results = pl.load(f)
results.keys()

In [ ]:
import pickle as pl
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_project/timeseries_with_chronos/training_curves_3d_focal/fold_probabilities.pkl','rb') as f:
    fold_probabilities = pl.load(f)

In [ ]:
# _save_cumulative_roc_added(fold_probabilities['y_true_folds'],fold_probabilities['y_score_folds'], "./", 5)
# _save_cumulative_pr_added(fold_probabilities['y_true_folds'],fold_probabilities['y_score_folds'], "./", 5)
# _save_cumulative_confusion_matrix(fold_probabilities['y_true_folds'], fold_probabilities['y_score_folds'], "./", 0.5)
